# 05 · Validación biológica, nulos de topología y DepMap

Este es el único notebook de validación biológica. Las anotaciones no se
usaron durante el ajuste. El análisis primario elimina PAM50 y conjuntos
que nombran las cohortes de desarrollo, cruza cada grafo con las cuatro
clases y usa 1.000 sorteos dentro del prior emparejados por tipo, signo y
grado de fuente/diana. Un segundo nulo conserva exactamente grado dirigido,
tipo y signo.

Esto sustituye al antiguo notebook 9 porque la hipótesis prioritaria es si
la aparente biología se explica por hubs/fuentes. No es una permutación de
etiquetas y no se describirá como tal.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "intergate").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Ejecute este notebook dentro de InterGATE_V8.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_v8_protocol, assert_transport_protocol, prepare_runtime,
)

assert_v8_protocol(CFG)
assert_transport_protocol(CFG)
ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / "INTERGATE_V8"
RESULTS = PROJECT_ROOT / "results_v8"
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
import re, hashlib, warnings
import requests
from scipy import stats

from intergate.biology_validation import (
    bh_adjust, candidate_edge_table, class_specificity_matrix,
    load_gmt, matched_edge_signature_null,
)
from intergate.reviewer_analysis import (
    enrichment_for_panels, rewiring_coherence_null,
)

runtime = prepare_runtime(CFG, include_external=False)
OUT = RESULTS / "05_biology_validation"
OUT.mkdir(parents=True, exist_ok=True)
LOCKED_BIOLOGY = (RESULTS/ "04_controls" / "locked_biology_score_w0075")

BIO_AGG = LOCKED_BIOLOGY / "biology_consensus"
PANEL_FILE = LOCKED_BIOLOGY / "biology_panels_locked.json"
RESOURCE_DIR = PROJECT_ROOT / "resources_biological"
MSIGDB = RESOURCE_DIR / "msigdb"
DEPMAP = Path(os.environ.get("INTERGATE_DEPMAP_DIR", RESOURCE_DIR / "depmap"))
MSIGDB.mkdir(parents=True, exist_ok=True)
N_MATCHED_DRAWS = int(CFG.BIOLOGY_MATCHED_NULL_DRAWS)
N_REWIRING_DRAWS = int(CFG.BIOLOGY_REWIRING_NULL_DRAWS)
RUN_DEPMAP = True


In [ ]:
def download_if_missing(url, destination):
    destination = Path(destination)
    if destination.is_file() and destination.stat().st_size > 1000:
        return
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    destination.write_bytes(response.content)
    if destination.stat().st_size <= 1000:
        raise RuntimeError(f"Descarga vacía o incompleta: {url}")

release = "2026.1.Hs"
hallmark_file = MSIGDB / f"h.all.v{release}.symbols.gmt"
cgp_file = MSIGDB / f"c2.cgp.v{release}.symbols.gmt"
base_url = f"https://data.broadinstitute.org/gsea-msigdb/msigdb/release/{release}"
download_if_missing(base_url + "/" + hallmark_file.name, hallmark_file)
download_if_missing(base_url + "/" + cgp_file.name, cgp_file)
hallmark = load_gmt(hallmark_file)
cgp = load_gmt(cgp_file)
pam50 = {
    line.strip().upper() for line in
    (RESOURCE_DIR / "PAM50_symbols.txt").read_text(encoding="utf-8").splitlines()
    if line.strip()
}




# Contrato fijo y direccional del manuscrito. Mantener exactamente la
# misma familia de anotación evita que la comparación V7/V8 mezcle un
# cambio de arquitectura con un cambio de firmas biológicas.
from intergate.revision_analyses import PRIMARY_SIGNATURE_TERMS

excluded_tokens = tuple(name.upper() for name in CFG.TRAIN_COHORTS)
selected_terms = {
    class_name: list(PRIMARY_SIGNATURE_TERMS[class_name])
    for class_name in runtime["classes"]
}
cohort_named_terms = [
    term
    for terms in selected_terms.values()
    for term in terms
    if any(token in term.upper() for token in excluded_tokens)
]
if cohort_named_terms:
    raise RuntimeError(
        "Una firma bloqueada menciona una cohorte de desarrollo: "
        + ", ".join(cohort_named_terms)
    )
signatures_with_pam50, signatures_primary = {}, {}
for class_name, terms in selected_terms.items():
    absent = [term for term in terms if term not in cgp]
    if absent:
        raise KeyError(
            f"MSigDB {release} no contiene los términos bloqueados "
            f"para {class_name}: {absent}"
        )
    members = set().union(*(cgp[term] for term in terms))
    signatures_with_pam50[class_name] = {
        str(gene).upper() for gene in members
    }
    signatures_primary[class_name] = {
        str(gene).upper()
        for gene in members
        if str(gene).upper() not in pam50
    }
    if len(signatures_primary[class_name]) < 10:
        raise RuntimeError(f"Firma no-PAM50 demasiado pequeña para {class_name}.")

signature_manifest = {
    "msigdb_release": release,
    "primary_excludes_pam50": True,
    "development_cohort_tokens_excluded": list(excluded_tokens),
    "selection_rule": "fixed_directionally_aligned_manuscript_terms",
    "selection_rule": "fixed_directionally_aligned_manuscript_terms",
    "selection_rule": "fixed_directionally_aligned_manuscript_terms",
    "terms": selected_terms,
    "sizes_primary": {key: len(value) for key, value in signatures_primary.items()},
    "external_opened": False,
}
(OUT / "locked_signature_manifest.json").write_text(
    json.dumps(signature_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(pd.DataFrame({
    "class_name": runtime["classes"],
    "n_terms": [len(selected_terms[c]) for c in runtime["classes"]],
    "n_genes_no_PAM50": [len(signatures_primary[c]) for c in runtime["classes"]],
}))


In [ ]:
graphs = {}
for class_name in runtime["classes"]:
    safe = "".join(c if c.isalnum() or c in "-_" else "_" for c in class_name)
    path = BIO_AGG / f"biology_consensus_{safe}.csv"
    graph = pd.read_csv(path)
    graph["source_gene"] = graph.source_gene.astype(str).str.upper()
    graph["target_gene"] = graph.target_gene.astype(str).str.upper()
    graph["sign"] = np.sign(pd.to_numeric(graph.prior_weight)).astype(int)
    graphs[class_name] = graph

candidates = candidate_edge_table(
    runtime["edge_index"], runtime["edge_weight"], runtime["edge_type"],
    runtime["genes"], n_degree_bins=10,
)
candidates.to_csv(OUT / "candidate_prior_for_null.csv.gz", index=False, compression="gzip")

summaries, distributions = [], []
for graph_id, class_name in enumerate(runtime["classes"]):
    print(graph_id)
    summary, distribution = matched_edge_signature_null(
        graphs[class_name], candidates, signatures_primary,
        n_draws=N_MATCHED_DRAWS, seed=20260819 + graph_id,
    )
    summary.insert(0, "graph_class", class_name)
    distribution.insert(0, "graph_class", class_name)
    summaries.append(summary)
    distributions.append(distribution)
matched_summary = pd.concat(summaries, ignore_index=True)
matched_summary["fdr_global_48_tests"] = bh_adjust(matched_summary.empirical_p)
matched_distribution = pd.concat(distributions, ignore_index=True)
matched_summary.to_csv(OUT / "matched_candidate_signature_tests.csv", index=False)
matched_distribution.to_csv(
    OUT / "matched_candidate_signature_distribution.csv.gz",
    index=False, compression="gzip",
)
specificity = class_specificity_matrix(graphs, signatures_primary)
specificity.to_csv(OUT / "cross_class_specificity.csv", index=False)
display(specificity.drop_duplicates("graph_class"))


In [ ]:
rewiring_rows = []
for graph_id, class_name in enumerate(runtime["classes"]):
    null = rewiring_coherence_null(
        graphs[class_name], hallmark,
        n_repeats=N_REWIRING_DRAWS,
        nswap_multiplier=10,
        seed=303000 + graph_id * 1000,
    )
    null.insert(0, "graph_class", class_name)
    null["empirical_p"] = float(null.attrs.get("empirical_p", np.nan))
    rewiring_rows.append(null)
rewiring = pd.concat(rewiring_rows, ignore_index=True)
changed_rewires = rewiring.loc[rewiring.kind.eq("rewired"), "changed_edge_rows"]
if changed_rewires.fillna(0).median() <= 0:
    raise RuntimeError(
        "El rewiring no cambió aristas; no puede usarse como nulo."
    )
rewiring.to_csv(OUT / "directed_degree_type_sign_rewiring.csv", index=False)
rewiring_summary = (
    rewiring.groupby("graph_class", as_index=False)
    .agg(
        empirical_p=("empirical_p", "first"),
        median_accepted_swaps=("accepted_swaps", "median"),
        median_changed_edge_rows=("changed_edge_rows", "median"),
    )
)
rewiring_summary["fdr_four_graphs"] = bh_adjust(rewiring_summary.empirical_p)
rewiring_summary.to_csv(OUT / "rewiring_coherence_tests.csv", index=False)

panels_payload = json.loads(PANEL_FILE.read_text(encoding="utf-8"))
panels = {}
for class_name, frame in graphs.items():
    panels[f"biology_graph_nodes::{class_name}"] = (
        set(frame.source_gene) | set(frame.target_gene)
    )
    strict = panels_payload["strict_5of5"][class_name]
    if strict:
        panels[f"strict_5of5::{class_name}"] = set(strict)
ora = enrichment_for_panels(
    panels,
    universe=runtime["genes"],
    collections={"Hallmark": hallmark, "C2_CGP": cgp},
    min_overlap=2,
    return_reportable_only=False,
)
ora.to_csv(OUT / "hallmark_c2_ora_complete_family.csv.gz", index=False, compression="gzip")
display(ora[(ora.fdr < 0.05) & ora.meets_min_overlap].head(30))


## DepMap (soporte ortogonal)

LumA y LumB se contrastan con líneas receptor-positivas,
HER2-enriched con ERBB2-altas y Basal-like con receptor-negativas. Son
proxies celulares, no equivalencias PAM50. Si la release no trae una
anotación molecular suficiente, la regla ESR1/PGR/ERBB2 se calcula una vez
y sus umbrales quedan en el manifiesto.


In [ ]:
def normalize_gene(value):
    return str(value).split(" (")[0].strip().upper()

def find_depmap_file(names):
    for name in names:
        path = DEPMAP / name
        if path.is_file():
            return path
    return DEPMAP / names[0]

def read_depmap_expression(path):
    table = pd.read_csv(path, low_memory=False)
    id_column = next(
        (name for name in ("ModelID", "DepMap_ID", "depmap_id") if name in table.columns),
        None,
    )
    if id_column is None:
        table = table.set_index(table.columns[0])
    else:
        if "is_default_entry" in table.columns:
            flag = table["is_default_entry"]
            if flag.dtype == object:
                flag = flag.astype(str).str.lower().isin({"true", "1", "yes"})
            table = table[flag.fillna(False).astype(bool)]
        metadata = [
            name for name in (
                "ProfileID", "profile_id", "is_default_entry",
                "DepMap_ID", "depmap_id",
            )
            if name in table.columns and name != id_column
        ]
        table = table.drop(columns=metadata, errors="ignore")
        table = table.drop_duplicates(id_column).set_index(id_column)
    table.index = table.index.astype(str)
    table.columns = [normalize_gene(value) for value in table.columns]
    table = table.loc[:, ~pd.Index(table.columns).duplicated()]
    return table.apply(pd.to_numeric, errors="coerce")

gene_effect_file = DEPMAP / "CRISPRGeneEffect.csv"
model_file = find_depmap_file(["Model.csv", "Models.csv"])
expression_file = find_depmap_file([
    "OmicsExpressionProteinCodingGenesTPMLogp1.csv",
    "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
])
if RUN_DEPMAP:
    missing = [path for path in (gene_effect_file, model_file) if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            "Faltan recursos DepMap. Consulte resources_biological/README.md:\n- "
            + "\n- ".join(map(str, missing))
        )

    gene_effect = pd.read_csv(gene_effect_file, index_col=0, low_memory=False)
    gene_effect.index = gene_effect.index.astype(str)
    gene_effect.columns = [normalize_gene(value) for value in gene_effect.columns]
    gene_effect = gene_effect.loc[:, ~pd.Index(gene_effect.columns).duplicated()]
    gene_effect = gene_effect.apply(pd.to_numeric, errors="coerce")

    models = pd.read_csv(model_file, low_memory=False)
    id_col = next((c for c in ("ModelID", "DepMap_ID", "depmap_id") if c in models), None)
    lineage_col = next((c for c in ("OncotreeLineage", "lineage", "primary_disease", "PrimaryDisease") if c in models), None)
    if id_col is None or lineage_col is None:
        raise KeyError("Model.csv requiere ModelID y un linaje reconocido.")
    models[id_col] = models[id_col].astype(str)
    breast = models[
        models[lineage_col].astype(str).str.contains("breast", case=False, na=False)
        & models[id_col].isin(gene_effect.index)
    ].copy()
    if breast.empty:
        raise RuntimeError("No hay líneas de mama con Chronos.")

    def map_group(value):
        text = str(value).lower()
        if "her2" in text or "erbb2" in text:
            return "her2_amplified"
        if any(token in text for token in ("basal", "triple", "tnbc", "receptor_negative")):
            return "receptor_negative"
        if any(token in text for token in ("luminal", "er+", "er_positive", "receptor_positive")):
            return "receptor_positive"
        return None

    molecular_col = next((c for c in (
        "lineage_molecular_subtype", "MolecularSubtype",
        "OncotreeSubtype", "OncotreeSubtypeName",
    ) if c in breast), None)
    subtype = pd.Series(dtype=object)
    grouping_method, grouping_thresholds = None, {}
    if molecular_col is not None:
        candidate = pd.Series(
            breast[molecular_col].map(map_group).to_numpy(),
            index=breast[id_col].to_numpy(), dtype=object,
        ).dropna()
        if all(candidate.value_counts().get(group, 0) >= 5 for group in (
            "receptor_positive", "her2_amplified", "receptor_negative"
        )):
            subtype = candidate
            grouping_method = f"Model.csv:{molecular_col}"

    if subtype.empty:
        if not expression_file.is_file():
            raise FileNotFoundError(
                "La anotación molecular es insuficiente y falta " + str(expression_file)
            )
        expression = read_depmap_expression(expression_file)
        receptor = expression.reindex(breast[id_col].astype(str))[ ["ESR1", "PGR", "ERBB2"] ].apply(pd.to_numeric, errors="coerce").dropna()
        grouping_thresholds = {
            "ESR1_median": float(receptor.ESR1.median()),
            "PGR_median": float(receptor.PGR.median()),
            "ERBB2_q75": float(receptor.ERBB2.quantile(0.75)),
        }
        her2 = receptor.ERBB2.gt(grouping_thresholds["ERBB2_q75"])
        positive = receptor.ESR1.gt(grouping_thresholds["ESR1_median"]) | receptor.PGR.gt(grouping_thresholds["PGR_median"])
        subtype = pd.Series(np.where(
            her2, "her2_amplified",
            np.where(positive, "receptor_positive", "receptor_negative"),
        ), index=receptor.index)
        grouping_method = "ESR1_or_PGR_median_and_ERBB2_q75_within_breast"
    subtype = subtype[subtype.index.isin(gene_effect.index)]
    group_counts = subtype.value_counts()
    if any(group_counts.get(group, 0) < 5 for group in (
        "receptor_positive", "her2_amplified", "receptor_negative"
    )):
        raise RuntimeError(f"DepMap infrapotenciado: {group_counts.to_dict()}")
    display(group_counts)


In [ ]:
if RUN_DEPMAP:
    essentiality = {}
    for group in ("receptor_positive", "her2_amplified", "receptor_negative"):
        inside = gene_effect.loc[gene_effect.index.intersection(subtype.index[subtype.eq(group)])]
        outside = gene_effect.loc[gene_effect.index.intersection(subtype.index[~subtype.eq(group)])]
        usable = inside.notna().sum().ge(3) & outside.notna().sum().ge(3)
        inside, outside = inside.loc[:, usable], outside.loc[:, usable]
        t_value, p_value = stats.ttest_ind(inside, outside, equal_var=False, nan_policy="omit")
        table = pd.DataFrame({
            "gene": inside.columns,
            "mean_inside": inside.mean().to_numpy(),
            "mean_outside": outside.mean().to_numpy(),
            "p": np.nan_to_num(p_value, nan=1.0),
        })
        table["delta"] = table.mean_inside - table.mean_outside
        table["fdr"] = bh_adjust(table.p)
        table["selective_primary"] = table.delta.le(-0.25) & table.mean_inside.le(-0.5) & table.fdr.lt(0.05)
        essentiality[group] = table.set_index("gene")
        table.to_csv(OUT / f"depmap_gene_essentiality_{group}.csv", index=False)

    universe = sorted(set(runtime["genes"]) & set(gene_effect.columns))
    endpoint_degree = pd.concat([candidates.source_gene, candidates.target_gene]).value_counts().reindex(universe).fillna(0)
    cuts = np.unique(np.quantile(endpoint_degree, np.linspace(0, 1, 11)))
    degree_bin = pd.Series(
        np.searchsorted(cuts[1:-1], endpoint_degree.to_numpy(), side="right"),
        index=universe,
    )
    pools = {value: degree_bin.index[degree_bin.eq(value)].to_numpy() for value in sorted(degree_bin.unique())}
    rng = np.random.default_rng(20260819)

    def matched_gene_draws(genes, n_draws=1000):
        genes = sorted(set(genes) & set(universe))
        bins = degree_bin.reindex(genes).value_counts()
        draws = []
        for _ in range(n_draws):
            draw = []
            for bin_id, count in bins.items():
                draw.extend(rng.choice(pools[bin_id], int(count), replace=False))
            draws.append(draw)
        return genes, draws

    group_for_class = {
        "LumA": "receptor_positive", "LumB": "receptor_positive",
        "HER2-enriched": "her2_amplified", "Basal-like": "receptor_negative",
    }
    strict_panels = panels_payload["strict_5of5"]
    depmap_panels = []
    for class_name, frame in graphs.items():
        depmap_panels.append((
            class_name, "consensus_network_nodes",
            sorted(set(frame.source_gene) | set(frame.target_gene)),
        ))
        depmap_panels.append((
            class_name, "strict_gene_core_5of5",
            strict_panels.get(class_name, []),
        ))
    depmap_rows = []
    for class_name, panel_type, genes in depmap_panels:
        genes, draws = matched_gene_draws(genes, 1000)
        if len(genes) < 3:
            warnings.warn(
                f"{class_name}/{panel_type}: <3 genes en DepMap; "
                "se marca no evaluable y no se sustituye."
            )
            continue
        for group, table in essentiality.items():
            delta = table.delta.reindex(universe)
            selective = table.selective_primary.reindex(universe).fillna(False)
            obs_delta = float(delta.reindex(genes).mean())
            null_delta = np.asarray([delta.reindex(draw).mean() for draw in draws], float)
            obs_selective = int(selective.reindex(genes).sum())
            null_selective = np.asarray([selective.reindex(draw).sum() for draw in draws], float)
            depmap_rows.append({
                "class_name": class_name, "panel_type": panel_type,
                "depmap_group": group,
                "is_focal_proxy": group_for_class[class_name] == group,
                "n_genes": len(genes), "mean_delta": obs_delta,
                "mean_delta_null": float(np.nanmean(null_delta)),
                "p_delta_lower": float((1 + np.sum(null_delta <= obs_delta)) / (1 + len(null_delta))),
                "n_selective": obs_selective,
                "n_selective_null": float(null_selective.mean()),
                "p_selective_upper": float((1 + np.sum(null_selective >= obs_selective)) / (1 + len(null_selective))),
            })
    depmap_results = pd.DataFrame(depmap_rows)
    if depmap_results.empty:
        raise RuntimeError("Ningún núcleo estricto es evaluable en DepMap.")
    depmap_results["fdr_delta"] = bh_adjust(depmap_results.p_delta_lower)
    depmap_results["fdr_selective"] = bh_adjust(depmap_results.p_selective_upper)
    depmap_results.to_csv(OUT / "depmap_strict_core_cross_group.csv", index=False)
    display(depmap_results)

    def sha256(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1 << 20), b""):
                digest.update(block)
        return digest.hexdigest()
    depmap_manifest = {
        "status": "complete", "grouping_method": grouping_method,
        "grouping_thresholds": grouping_thresholds,
        "group_counts": {str(k): int(v) for k, v in group_counts.items()},
        "proxies_are_not_pam50_equivalents": True,
        "resource_sha256": {
            str(gene_effect_file): sha256(gene_effect_file),
            str(model_file): sha256(model_file),
            str(expression_file): sha256(expression_file) if expression_file.is_file() else None,
        },
        "external_transcriptomic_test_opened": False,
    }
    (OUT / "depmap_manifest.json").write_text(
        json.dumps(depmap_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
print("Validación biológica V8 completa:", OUT)
